In [ ]:
import requests
import time
from mal_scraper import get_user_anime_list
import logging
import pandas as pd
import os
from pathlib import Path
import mal_scraper.users
from mal_scraper.consts import ConsumptionStatus
import numpy as np
from sklearn.decomposition import TruncatedSVD
import joblib
import json
import scipy.sparse as sp

In [2]:
logging.getLogger('jikanpy').setLevel(logging.CRITICAL)
logging.getLogger().setLevel(logging.CRITICAL)

In [3]:
delay = 1
maxUserSize = 2000

In [4]:
def getRandomUser():
    time.sleep(delay)
    url = "https://api.tenrai.org/v1/random/users"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            return data['data']
        elif response.status_code == 429:
            print("Rate limited! Waiting before retry...")
            time.sleep(0.1)
            return getRandomUser()
        else:
            print(f"Error: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"An error occurred: {e}")
        return None

In [5]:
url = "https://api.tenrai.org/v1/top/anime"
while True:
    response = requests.get(url)
    
    time.sleep(delay)
    
    data = response.json()
    if not 'status' in data or data['status'] != 500:
        break


In [6]:
"""
url = "https://api.tenrai.org/v1/top/anime"
popular_anime_ids = []
for page in range(1, 100):
    params = {
        "filter": "bypopularity",
        "page": page,
    }
    response = requests.get(url, params=params)
    time.sleep(delay)
    data = response.json()
    # print(data)
    animes = data['data']
    popular_anime_ids += [anime['mal_id'] for anime in animes]

print(popular_anime_ids)
print(len(popular_anime_ids))
"""

'\nurl = "https://api.tenrai.org/v1/top/anime"\npopular_anime_ids = []\nfor page in range(1, 100):\n    params = {\n        "filter": "bypopularity",\n        "page": page,\n    }\n    response = requests.get(url, params=params)\n    time.sleep(delay)\n    data = response.json()\n    # print(data)\n    animes = data[\'data\']\n    popular_anime_ids += [anime[\'mal_id\'] for anime in animes]\n\nprint(popular_anime_ids)\nprint(len(popular_anime_ids))\n'

In [7]:
def get_anime_reviews_tenrai(anime_id, page):
    url = f"https://api.tenrai.org/v1/anime/{anime_id}/reviews?page={page}"
    # print(url)
    
    response = requests.get(url)
    time.sleep(delay)
    
    if response.status_code == 200:
        return response.json()
    else:
        return None

In [8]:
number = 1
t0 = time.time()
distinctUsers = set()
start = 0

In [9]:
file_path = Path.cwd() / "usernames.txt"
print(file_path)
    
with open(file_path, 'r') as f:
    for line in f:
        user = line.strip()
        if user:
            distinctUsers.add(user)

start += 1

print(f"Found {len(distinctUsers)} unique users.")

c:\dev\anime_checker\src-tauri\resources\usernames.txt
Found 11476 unique users.


In [10]:
"""
for ind in range(start, len(popular_anime_ids)):
    animeId = popular_anime_ids[ind]
    # print(animeId)
    page = 1
    while True:
        # print()
    # for page in range(1, 10):        
        data = get_anime_reviews_tenrai(animeId, page)
        # print("AAA", page, data)
        # print(data)

        if not data or ('status' in data and data['status'] == 500):
            page += 1
            continue

        number_of_failed_attempts = 0
            
        for i, review in enumerate(data['data'], 1):
            # print(f"\nReview {i}:")
            # print(f"User: {review['user']['username']}")
            # print(f"Score: {review['score']}")
            distinctUsers.add(review['user']['username'])

        if not data['pagination']['has_next_page']:
            print(f"{number} ended at: {page} with total of {len(distinctUsers)} users after {time.time() - t0} seconds")
            break
        page += 1
    number += 1
    # if len(distinctUsers) >= maxUserSize:
    #    print(f"Ending with {len(distinctUsers)}")
    #    break

start = ind
"""

'\nfor ind in range(start, len(popular_anime_ids)):\n    animeId = popular_anime_ids[ind]\n    # print(animeId)\n    page = 1\n    while True:\n        # print()\n    # for page in range(1, 10):        \n        data = get_anime_reviews_tenrai(animeId, page)\n        # print("AAA", page, data)\n        # print(data)\n\n        if not data or (\'status\' in data and data[\'status\'] == 500):\n            page += 1\n            continue\n\n        number_of_failed_attempts = 0\n\n        for i, review in enumerate(data[\'data\'], 1):\n            # print(f"\nReview {i}:")\n            # print(f"User: {review[\'user\'][\'username\']}")\n            # print(f"Score: {review[\'score\']}")\n            distinctUsers.add(review[\'user\'][\'username\'])\n\n        if not data[\'pagination\'][\'has_next_page\']:\n            print(f"{number} ended at: {page} with total of {len(distinctUsers)} users after {time.time() - t0} seconds")\n            break\n        page += 1\n    number += 1\n    # 

In [11]:
print(len(distinctUsers))

11476


In [12]:
usernames = list(distinctUsers)

In [13]:
print(len(set(usernames)))

11476


In [14]:
# usernames = usernames[:100]

In [15]:
_original_convert = mal_scraper.users._convert_status_code_to_const

def _safe_convert_status_code_to_const(code):
    status_map = {
        1: ConsumptionStatus.consuming,
        2: ConsumptionStatus.completed,
        3: ConsumptionStatus.on_hold,
        4: ConsumptionStatus.dropped,
        6: ConsumptionStatus.backlog,
    }
    return status_map.get(code, ConsumptionStatus.backlog)

mal_scraper.users._convert_status_code_to_const = _safe_convert_status_code_to_const

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://myanimelist.net/",
})

In [16]:
def getReviewsFromUser(username):
    anime_list = get_user_anime_list(username, requester=session)
    time.sleep(delay)
    if anime_list is None: return None
    anime_list = [anime for anime in anime_list if anime['score'] > 0]
    return anime_list

In [17]:
rows = []
rowsAtLeast5 = []
successful_usernames = []
usernamesAtLeast5 = []
count = 0

for username in usernames:
    try:
        count += 1
        animeList = getReviewsFromUser(username)
        
        if animeList is None:
            print(f"Skipping {username}: No data returned.")
            continue

        shortedAnimeList = [
            {
                'id_ref': entry['id_ref'],
                'name': entry['name'],
                'score': entry['score'],
            } 
            for entry in animeList
        ]
        # print(animeList)
        # print(shortedAnimeList)
        # print(len(animeList))

        rows.append(shortedAnimeList)
        successful_usernames.append(username)
        
        print(f"Processed: {count} users")
        if len(shortedAnimeList) > 4:
            rowsAtLeast5.append(shortedAnimeList)
            usernamesAtLeast5.append(username)

    except Exception as e:
        print(f"Error processing {username}: {e}")
        continue

Processed: 1 users
Processed: 2 users
Processed: 3 users
Processed: 4 users
Processed: 5 users
Processed: 6 users
Processed: 7 users
Processed: 8 users
Processed: 9 users
Processed: 10 users
Processed: 11 users
Processed: 12 users
Processed: 13 users
Processed: 14 users
Processed: 15 users
Processed: 16 users
Processed: 17 users
Processed: 18 users
Skipping JacksonDruker: No data returned.
Processed: 20 users
Processed: 21 users
Processed: 22 users
Processed: 23 users
Processed: 24 users
Processed: 25 users
Processed: 26 users
Processed: 27 users
Processed: 28 users
Processed: 29 users
Processed: 30 users
Processed: 31 users
Processed: 32 users
Processed: 33 users
Processed: 34 users
Processed: 35 users
Processed: 36 users
Processed: 37 users
Processed: 38 users
Processed: 39 users
Processed: 40 users
Processed: 41 users
Processed: 42 users
Processed: 43 users
Processed: 44 users
Processed: 45 users
Skipping angecritter: No data returned.
Processed: 47 users
Processed: 48 users
Process

In [26]:
df = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rows
], index=successful_usernames)

df = df.fillna(0)

df

,52196_Date A Live V,59062_Gachiakuta,58488_Sengoku Youko: Senma Konton-hen,"58714_Saikyou no Shienshoku ""Wajutsushi"" de Aru Ore wa Sekai Saikyou Clan wo Shitagaeru",57100_The New Gate,53835_Unnamed Memory,56980_Karasu wa Aruji wo Erabanai,41457_86,48569_86 Part 2,59644_Yasei no Last Boss ga Arawareta!,...,57150_Creature Hunters,5117_Kokudo Ou,"56732_Sekai Saikou no Ansatsusha, Isekai Kizoku ni Tensei suru Season 2",51185_Nonsense Bungaku (2022),10040_Chinpui: Eri-sama Katsudou Daishashin,63293_Hotaru no Yomeiri,"60948_Mezametara Saikyou Soubi to Uchuusenmochi Datta node, Ikkodate Mezashite Youhei toshite Jiyuu ni Ikitai",58873_Mark Your Kiss The Animation,50039_Tabun,59204_Magic Knight Rayearth (2026)
RUZIEL,9.0,8.0,9.0,8.0,8.0,7.0,9.0,9.0,10.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Wewel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KodySapphire,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
bentleys,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
OrochiMaurya,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
anabrent,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Swicce,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dANGERJet409,6.0,10.0,0.0,10.0,4.0,7.0,0.0,10.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
salimus,0.0,0.0,0.0,0.0,0.0,0.0,7.0,10.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [27]:
df2 = pd.DataFrame([
    {f"{item['id_ref']}_{item['name']}": item['score'] for item in userAnime}
    for userAnime in rowsAtLeast5
], index=usernamesAtLeast5)

df2 = df2.fillna(0)

df2

,52196_Date A Live V,59062_Gachiakuta,58488_Sengoku Youko: Senma Konton-hen,"58714_Saikyou no Shienshoku ""Wajutsushi"" de Aru Ore wa Sekai Saikyou Clan wo Shitagaeru",57100_The New Gate,53835_Unnamed Memory,56980_Karasu wa Aruji wo Erabanai,41457_86,48569_86 Part 2,59644_Yasei no Last Boss ga Arawareta!,...,57150_Creature Hunters,5117_Kokudo Ou,"56732_Sekai Saikou no Ansatsusha, Isekai Kizoku ni Tensei suru Season 2",51185_Nonsense Bungaku (2022),10040_Chinpui: Eri-sama Katsudou Daishashin,63293_Hotaru no Yomeiri,"60948_Mezametara Saikyou Soubi to Uchuusenmochi Datta node, Ikkodate Mezashite Youhei toshite Jiyuu ni Ikitai",58873_Mark Your Kiss The Animation,50039_Tabun,59204_Magic Knight Rayearth (2026)
RUZIEL,9.0,8.0,9.0,8.0,8.0,7.0,9.0,9.0,10.0,8.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Wewel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
KodySapphire,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
bentleys,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
OrochiMaurya,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
anabrent,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Swicce,0.0,8.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dANGERJet409,6.0,10.0,0.0,10.0,4.0,7.0,0.0,10.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
salimus,0.0,0.0,0.0,0.0,0.0,0.0,7.0,10.0,10.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
anime_names = df.columns.tolist()
with open('anime_names.json', 'w', encoding='utf-8') as f:
    json.dump(anime_names, f)

In [ ]:
anime_names2 = df2.columns.tolist()
with open('anime_names2.json', 'w', encoding='utf-8') as f:
    json.dump(anime_names2, f)

In [ ]:
vals = df.values.astype(np.float32)
mask = vals != 0

row_sums = vals.sum(axis=1)
row_counts = mask.sum(axis=1)
row_means = np.divide(
    row_sums, 
    row_counts, 
    out=np.zeros(len(vals), dtype=np.float32), 
    where=row_counts != 0
)

vals_norm = np.where(mask, vals - row_means[:, np.newaxis], 0)

df_norm = pd.DataFrame(vals_norm, index=df.index, columns=df.columns)

X_sparse = sp.csr_matrix(vals_norm)

svd = TruncatedSVD(n_components=50, algorithm='randomized', random_state=42)
base_dense_vectors = svd.fit_transform(X_sparse)

joblib.dump(svd, 'svd_model.joblib')
np.save('svd_vectors.npy', base_dense_vectors.astype(np.float32))
df_norm.to_csv('df_norm.csv.gz', index=False, compression='gzip')
sp.save_npz('df_norm_sparse.npz', X_sparse)

In [ ]:
vals2 = df2.values.astype(np.float32)
mask2 = vals2 != 0

row_sums2 = vals2.sum(axis=1)
row_counts2 = mask2.sum(axis=1)
row_means2 = np.divide(
    row_sums2, 
    row_counts2, 
    out=np.zeros(len(vals2), dtype=np.float32), 
    where=row_counts2 != 0
)

vals_norm2 = np.where(mask2, vals2 - row_means2[:, np.newaxis], 0)

df_norm2 = pd.DataFrame(vals_norm2, index=df2.index, columns=df2.columns)

X_sparse2 = sp.csr_matrix(vals_norm2)

svd2 = TruncatedSVD(n_components=50, algorithm='randomized', random_state=42)
base_dense_vectors2 = svd2.fit_transform(X_sparse2)

joblib.dump(svd2, 'svd_model2.joblib')
np.save('svd_vectors2.npy', base_dense_vectors2.astype(np.float32))
df_norm2.to_csv('df_norm2.csv.gz', index=False, compression='gzip')
sp.save_npz('df_norm_sparse2.npz', X_sparse2)

In [ ]:
"""
df_ind = 1
while True:
    filename = f"anime_df_{df_ind}.csv"
    
    if not os.path.exists(filename):
        break
            
    df_ind += 1

df.to_csv(f"anime_df_{df_ind}.csv")
"""

'\ndf_ind = 1\nwhile True:\n    filename = f"anime_df_{df_ind}.csv"\n\n    if not os.path.exists(filename):\n        break\n\n    df_ind += 1\n\ndf.to_csv(f"anime_df_{df_ind}.csv")\n'

In [30]:
print(len(usernames))

11476


In [31]:
print(start)

1
